# Telugu Scriptio Continua Word Segmentation (2-State `STATE_2` Prediction)
## Abugida Syllable & Multi-Vottu Sub-Consonant Decomposition Framework

This notebook trains sequence tagger models (**BiLSTM**, **BiGRU**, **BiRNN**, **CNN**) on the Telugu dataset to predict `STATE_2` binary word boundaries (`0` = inner character, `1` = word boundary) from continuous text.

### Multi-Vottu Telugu Abugida Feature Representation
To handle complex Telugu/Sanskrit conjuncts with **multiple sub-consonants (వొత్తులు - Vottulu)** like `స్త్ర`, `స్ట్రా`, `డ్జ్ను`, `క్ష్మ`, each Telugu letter/Akshara is decomposed into **7 categorical feature slots**:
1. **Base Character / Consonant / Independent Vowel / Digit** (`base_id`)
2. **Virama (పొల్లు - Pollu)** (`virama_id`)
3. **Primary Sub-consonant (Vottu 1)** (`vottu1_id`)
4. **Secondary Sub-consonant (Vottu 2)** (`vottu2_id`)
5. **Tertiary Sub-consonant (Vottu 3)** (`vottu3_id`)
6. **Vowel Sign (మాత్ర - Maatra)** (`maatra_id`)
7. **Anusvara (సున్నా - Sunna)** / Visarga / Candrabindu (`sunna_id`)

Each component is embedded via a 64-dimensional trainable embedding layer, combined, and passed to the neural taggers to predict **1 label per Telugu letter**.

### Kaggle Execution & Outputs Saved
- **Dataset Path**: `/kaggle/input/datasets/vallurikeerthiram/testing`
- **Saved Model PKLs**: `telugu_2state_bilstm.pkl`, `telugu_2state_gru.pkl`, `telugu_2state_rnn.pkl`, `telugu_2state_cnn.pkl`, `telugu_2state_vocab.pkl`
- **Excel Predictions Output**: `telugu_2state_predictions.xlsx` containing `TRAIN_PREDICTIONS`, `VAL_PREDICTIONS`, `TEST_PREDICTIONS`.

In [1]:
# Step 1: Install required packages if needed
!pip install -q pandas openpyxl torch

In [2]:
from __future__ import annotations

import os
import random
import re
import sys
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

# ── Global Settings & Constants ───────────────────────────────────────────────
RANDOM_SEED = 42
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
NONE_TOKEN = "NONE"
PAD_LABEL = -100

AKSHARA_REGEX = re.compile(
    r'[అ-హౘ-ౚ0-9A-Za-z]'
    r'[ఁ-ఃా-ౌౕౖ]*'
    r'(?:్[క-హౘ-ౚ][ఁ-ఃా-ౌౕౖ]*)*'
    r'్?'
    r'[ఁ-ః]?'
    r'|.'
)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def select_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


@dataclass
class Config:
    data_dir: Path = Path("/kaggle/input/datasets/vallurikeerthiram/testing")
    local_fallback: Path = Path(".")
    excel_filename: str = "telugu_sentence_level_first_100_articles_split.xlsx"
    output_filename: str = "telugu_2state_predictions.xlsx"
    embedding_dim: int = 64
    hidden_dim: int = 128
    cnn_channels: int = 128
    dropout: float = 0.2
    batch_size: int = 64
    epochs: int = 8
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    patience: int = 2
    num_workers: int = 0


set_seed(RANDOM_SEED)
device = select_device()
print(f"PyTorch version: {torch.__version__}")
print(f"Target Execution Device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Count: {torch.cuda.device_count()}")

PyTorch version: 2.10.0+cu128
Target Execution Device: cuda
GPU Name: Tesla T4
Total GPU Count: 2


In [3]:
# Step 2: Akshara Tokenization & Multi-Vottu Decomposition Logic
def extract_aksharas(text: str) -> List[str]:
    words = str(text).split("\u200c")
    aksharas: List[str] = []
    for word in words:
        if word:
            aksharas.extend(AKSHARA_REGEX.findall(word))
    return aksharas


def decompose_akshara(akshara_str: str) -> Tuple[str, str, str, str, str, str, str]:
    """Decomposes an Akshara into 7 feature slots:
    (base, virama, vottu1, vottu2, vottu3, maatra, sunna)
    """
    base = ""
    has_virama = NONE_TOKEN
    vottus: List[str] = []
    maatra = NONE_TOKEN
    sunna = NONE_TOKEN

    chars = list(akshara_str)
    if not chars:
        return (PAD_TOKEN, NONE_TOKEN, NONE_TOKEN, NONE_TOKEN, NONE_TOKEN, NONE_TOKEN, NONE_TOKEN)

    base = chars[0]
    i = 1
    n = len(chars)

    while i < n:
        c = chars[i]
        code = ord(c)
        if code == 0x0C4D:  # Virama ( Pollu )
            has_virama = "VIRAMA"
            i += 1
            if i < n and ((0x0C15 <= ord(chars[i]) <= 0x0C39) or (0x0C58 <= ord(chars[i]) <= 0x0C5A)):
                vottus.append(chars[i])  # Sub-consonant ( Vottu )
                i += 1
        elif (0x0C3E <= code <= 0x0C4C) or (code in (0x0C55, 0x0C56)):
            maatra = c  # Vowel Sign ( Maatra )
            i += 1
        elif 0x0C01 <= code <= 0x0C03:
            sunna = c  # Anusvara ( Sunna / Visarga )
            i += 1
        else:
            i += 1

    vottu1 = vottus[0] if len(vottus) > 0 else NONE_TOKEN
    vottu2 = vottus[1] if len(vottus) > 1 else NONE_TOKEN
    vottu3 = vottus[2] if len(vottus) > 2 else NONE_TOKEN

    return (base, has_virama, vottu1, vottu2, vottu3, maatra, sunna)


def parse_state2(state_text: object) -> List[int]:
    compact = "".join(str(state_text).split())
    return [int(ch) for ch in compact if ch in ("0", "1")]


def reconstruct_sentence_from_state2(aksharas: Sequence[str], labels: Sequence[int]) -> str:
    words: List[str] = []
    curr: List[str] = []
    for ak, lbl in zip(aksharas, labels):
        curr.append(ak)
        if int(lbl) == 1:
            words.append("".join(curr))
            curr = []
    if curr:
        words.append("".join(curr))
    return " ".join(words)

In [5]:
# Step 3: Vocabulary Management & Multi-Vottu Datasets
@dataclass
class TeluguVocab:
    base_vocab: Dict[str, int]
    virama_vocab: Dict[str, int]
    vottu_vocab: Dict[str, int]
    maatra_vocab: Dict[str, int]
    sunna_vocab: Dict[str, int]

    @classmethod
    def build(cls, train_df: pd.DataFrame) -> "TeluguVocab":
        base_counts = Counter()
        vottu_counts = Counter()
        maatra_counts = Counter()
        sunna_counts = Counter()

        for script in train_df["scriptio_continua"]:
            aks = extract_aksharas(script)
            for ak in aks:
                base, _, v1, v2, v3, maatra, sunna = decompose_akshara(ak)
                base_counts[base] += 1
                for v in (v1, v2, v3):
                    if v != NONE_TOKEN:
                        vottu_counts[v] += 1
                if maatra != NONE_TOKEN:
                    maatra_counts[maatra] += 1
                if sunna != NONE_TOKEN:
                    sunna_counts[sunna] += 1

        def make_dict(counts: Counter, extra_tokens: List[str]) -> Dict[str, int]:
            v = {PAD_TOKEN: 0, UNK_TOKEN: 1}
            for token in extra_tokens:
                if token not in v:
                    v[token] = len(v)
            for token in sorted(counts):
                if token not in v:
                    v[token] = len(v)
            return v

        base_vocab = make_dict(base_counts, [])
        virama_vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1, NONE_TOKEN: 2, "VIRAMA": 3}
        vottu_vocab = make_dict(vottu_counts, [NONE_TOKEN])
        maatra_vocab = make_dict(maatra_counts, [NONE_TOKEN])
        sunna_vocab = make_dict(sunna_counts, [NONE_TOKEN])

        return cls(base_vocab, virama_vocab, vottu_vocab, maatra_vocab, sunna_vocab)


class TeluguAksharaDataset(Dataset):
    def __init__(self, df: pd.DataFrame, vocab: TeluguVocab) -> None:
        self.dataframe = df.reset_index(drop=True)
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, idx: int) -> Dict[str, object]:
        row = self.dataframe.iloc[idx]
        script = str(row["scriptio_continua"])
        state2 = row["STATE2_LIST"]
        aksharas = extract_aksharas(script)

        min_len = min(len(aksharas), len(state2))
        aksharas = aksharas[:min_len]
        labels = state2[:min_len]

        base_ids, virama_ids, vottu1_ids, vottu2_ids, vottu3_ids, maatra_ids, sunna_ids = (
            [],
            [],
            [],
            [],
            [],
            [],
            [],
        )

        for ak in aksharas:
            base, virama, v1, v2, v3, maatra, sunna = decompose_akshara(ak)
            base_ids.append(self.vocab.base_vocab.get(base, self.vocab.base_vocab[UNK_TOKEN]))
            virama_ids.append(self.vocab.virama_vocab.get(virama, self.vocab.virama_vocab[UNK_TOKEN]))
            vottu1_ids.append(self.vocab.vottu_vocab.get(v1, self.vocab.vottu_vocab[UNK_TOKEN]))
            vottu2_ids.append(self.vocab.vottu_vocab.get(v2, self.vocab.vottu_vocab[UNK_TOKEN]))
            vottu3_ids.append(self.vocab.vottu_vocab.get(v3, self.vocab.vottu_vocab[UNK_TOKEN]))
            maatra_ids.append(self.vocab.maatra_vocab.get(maatra, self.vocab.maatra_vocab[UNK_TOKEN]))
            sunna_ids.append(self.vocab.sunna_vocab.get(sunna, self.vocab.sunna_vocab[UNK_TOKEN]))

        return {
            "base_ids": torch.tensor(base_ids, dtype=torch.long),
            "virama_ids": torch.tensor(virama_ids, dtype=torch.long),
            "vottu1_ids": torch.tensor(vottu1_ids, dtype=torch.long),
            "vottu2_ids": torch.tensor(vottu2_ids, dtype=torch.long),
            "vottu3_ids": torch.tensor(vottu3_ids, dtype=torch.long),
            "maatra_ids": torch.tensor(maatra_ids, dtype=torch.long),
            "sunna_ids": torch.tensor(sunna_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "art_id": row.get("Art_id", row.get("ART_ID", "")),
            "para_id": row.get("para_id", row.get("PARA_ID", "")),
            "sent_id": row.get("Sent_id", row.get("SENT_ID", "")),
            "scriptio_continua": script,
            "sentence_gt": str(row["Sentence"]),
            "aksharas": aksharas,
        }


def collate_telugu_batch(batch: Sequence[Dict[str, object]]) -> Dict[str, object]:
    padded_base = pad_sequence([b["base_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_virama = pad_sequence([b["virama_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_v1 = pad_sequence([b["vottu1_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_v2 = pad_sequence([b["vottu2_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_v3 = pad_sequence([b["vottu3_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_maatra = pad_sequence([b["maatra_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_sunna = pad_sequence([b["sunna_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_labels = pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=PAD_LABEL)
    mask = padded_labels.ne(PAD_LABEL)

    return {
        "base_ids": padded_base,
        "virama_ids": padded_virama,
        "vottu1_ids": padded_v1,
        "vottu2_ids": padded_v2,
        "vottu3_ids": padded_v3,
        "maatra_ids": padded_maatra,
        "sunna_ids": padded_sunna,
        "labels": padded_labels,
        "mask": mask,
        "art_ids": [b["art_id"] for b in batch],
        "para_ids": [b["para_id"] for b in batch],
        "sent_ids": [b["sent_id"] for b in batch],
        "scriptio_continua": [b["scriptio_continua"] for b in batch],
        "sentence_gt": [b["sentence_gt"] for b in batch],
        "aksharas": [b["aksharas"] for b in batch],
    }

In [6]:
# Step 4: Multi-Vottu Feature Embedding & Neural Architectures
class TeluguAksharaEmbedding(nn.Module):
    def __init__(self, vocab: TeluguVocab, embed_dim: int = 64) -> None:
        super().__init__()
        self.emb_base = nn.Embedding(len(vocab.base_vocab), embed_dim, padding_idx=0)
        self.emb_virama = nn.Embedding(len(vocab.virama_vocab), embed_dim, padding_idx=0)
        self.emb_vottu = nn.Embedding(len(vocab.vottu_vocab), embed_dim, padding_idx=0)
        self.emb_maatra = nn.Embedding(len(vocab.maatra_vocab), embed_dim, padding_idx=0)
        self.emb_sunna = nn.Embedding(len(vocab.sunna_vocab), embed_dim, padding_idx=0)

        self.proj = nn.Linear(embed_dim * 7, embed_dim)
        self.act = nn.ReLU()

    def forward(
        self,
        base_ids: torch.Tensor,
        virama_ids: torch.Tensor,
        vottu1_ids: torch.Tensor,
        vottu2_ids: torch.Tensor,
        vottu3_ids: torch.Tensor,
        maatra_ids: torch.Tensor,
        sunna_ids: torch.Tensor,
    ) -> torch.Tensor:
        b_e = self.emb_base(base_ids)
        v_e = self.emb_virama(virama_ids)
        vt1_e = self.emb_vottu(vottu1_ids)
        vt2_e = self.emb_vottu(vottu2_ids)
        vt3_e = self.emb_vottu(vottu3_ids)
        m_e = self.emb_maatra(maatra_ids)
        s_e = self.emb_sunna(sunna_ids)

        concat = torch.cat([b_e, v_e, vt1_e, vt2_e, vt3_e, m_e, s_e], dim=-1)
        return self.act(self.proj(concat))


class BiLSTMTagger(nn.Module):
    def __init__(self, vocab: TeluguVocab, config: Config, num_labels: int = 2) -> None:
        super().__init__()
        self.embedding = TeluguAksharaEmbedding(vocab, config.embedding_dim)
        self.dropout = nn.Dropout(config.dropout)
        self.lstm = nn.LSTM(
            input_size=config.embedding_dim,
            hidden_size=config.hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Linear(config.hidden_dim * 2, num_labels)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        x = self.embedding(
            batch["base_ids"],
            batch["virama_ids"],
            batch["vottu1_ids"],
            batch["vottu2_ids"],
            batch["vottu3_ids"],
            batch["maatra_ids"],
            batch["sunna_ids"],
        )
        outputs, _ = self.lstm(self.dropout(x))
        return self.classifier(self.dropout(outputs))


class GRUTagger(nn.Module):
    def __init__(self, vocab: TeluguVocab, config: Config, num_labels: int = 2) -> None:
        super().__init__()
        self.embedding = TeluguAksharaEmbedding(vocab, config.embedding_dim)
        self.dropout = nn.Dropout(config.dropout)
        self.gru = nn.GRU(
            input_size=config.embedding_dim,
            hidden_size=config.hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Linear(config.hidden_dim * 2, num_labels)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        x = self.embedding(
            batch["base_ids"],
            batch["virama_ids"],
            batch["vottu1_ids"],
            batch["vottu2_ids"],
            batch["vottu3_ids"],
            batch["maatra_ids"],
            batch["sunna_ids"],
        )
        outputs, _ = self.gru(self.dropout(x))
        return self.classifier(self.dropout(outputs))


class RNNTagger(nn.Module):
    def __init__(self, vocab: TeluguVocab, config: Config, num_labels: int = 2) -> None:
        super().__init__()
        self.embedding = TeluguAksharaEmbedding(vocab, config.embedding_dim)
        self.dropout = nn.Dropout(config.dropout)
        self.rnn = nn.RNN(
            input_size=config.embedding_dim,
            hidden_size=config.hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            nonlinearity="tanh",
        )
        self.classifier = nn.Linear(config.hidden_dim * 2, num_labels)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        x = self.embedding(
            batch["base_ids"],
            batch["virama_ids"],
            batch["vottu1_ids"],
            batch["vottu2_ids"],
            batch["vottu3_ids"],
            batch["maatra_ids"],
            batch["sunna_ids"],
        )
        outputs, _ = self.rnn(self.dropout(x))
        return self.classifier(self.dropout(outputs))


class CNNTagger(nn.Module):
    def __init__(self, vocab: TeluguVocab, config: Config, num_labels: int = 2) -> None:
        super().__init__()
        self.embedding = TeluguAksharaEmbedding(vocab, config.embedding_dim)
        self.dropout = nn.Dropout(config.dropout)
        self.conv = nn.Conv1d(config.embedding_dim, config.cnn_channels, kernel_size=3, padding=1)
        self.activation = nn.ReLU()
        self.classifier = nn.Linear(config.cnn_channels, num_labels)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        x = self.dropout(
            self.embedding(
                batch["base_ids"],
                batch["virama_ids"],
                batch["vottu1_ids"],
                batch["vottu2_ids"],
                batch["vottu3_ids"],
                batch["maatra_ids"],
                batch["sunna_ids"],
            )
        ).transpose(1, 2)
        x = self.activation(self.conv(x)).transpose(1, 2)
        return self.classifier(self.dropout(x))


def build_model(model_name: str, vocab: TeluguVocab, config: Config) -> nn.Module:
    name = model_name.lower()
    if name == "bilstm":
        return BiLSTMTagger(vocab, config)
    if name == "gru":
        return GRUTagger(vocab, config)
    if name == "rnn":
        return RNNTagger(vocab, config)
    if name == "cnn":
        return CNNTagger(vocab, config)
    raise ValueError(f"Unknown model name: {model_name}")

In [7]:
# Step 5: Training & Inference Functions
def train_one_epoch(
    model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, device: torch.device
) -> float:
    model.train()
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_LABEL)
    total_loss = 0.0

    for batch in loader:
        b_dev = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        labels = b_dev["labels"]
        optimizer.zero_grad()
        logits = model(b_dev)
        loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / max(len(loader), 1)


def predict_and_generate_rows(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    model_name: str,
    split_name: str,
) -> pd.DataFrame:
    model.eval()
    rows: List[Dict[str, object]] = []

    with torch.no_grad():
        for batch in loader:
            b_dev = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            labels = b_dev["labels"]
            mask = b_dev["mask"]

            logits = model(b_dev)
            preds = logits.argmax(dim=-1)

            batch_size_actual = len(batch["art_ids"])
            for idx in range(batch_size_actual):
                seq_mask = mask[idx]
                true_seq = labels[idx][seq_mask].tolist()
                pred_seq = preds[idx][seq_mask].tolist()

                aksharas = batch["aksharas"][idx][: len(true_seq)]
                script = batch["scriptio_continua"][idx]
                gt_sentence = batch["sentence_gt"][idx]

                gt_labels_str = "".join(str(x) for x in true_seq)
                pred_labels_str = "".join(str(x) for x in pred_seq)

                predicted_sentence = reconstruct_sentence_from_state2(aksharas, pred_seq)

                rows.append(
                    {
                        "Art_id": batch["art_ids"][idx],
                        "para_id": batch["para_ids"][idx],
                        "Sent_id": batch["sent_ids"][idx],
                        "INPUT_SENTENCE": script,
                        "GROUND_TRUTH_SENTENCE": gt_sentence,
                        "PREDICTED_SENTENCE": predicted_sentence,
                        "GROUND_TRUTH_LABELS": gt_labels_str,
                        "PREDICTED_LABELS": pred_labels_str,
                        "MODEL": model_name.upper(),
                        "SPLIT": split_name.upper(),
                    }
                )

    return pd.DataFrame(rows)


def fit_and_evaluate_model(
    model_name: str,
    vocab: TeluguVocab,
    loaders: Dict[str, DataLoader],
    device: torch.device,
    config: Config,
) -> Dict[str, pd.DataFrame]:
    print(f"\n================ Training {model_name.upper()} ================")
    model = build_model(model_name, vocab, config).to(device)

    if torch.cuda.device_count() > 1:
        print(f"[Multi-GPU] Wrapping {model_name.upper()} with DataParallel across {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

    for epoch in range(1, config.epochs + 1):
        loss = train_one_epoch(model, loaders["train"], optimizer, device)
        print(f"[{model_name.upper()}] Epoch {epoch}/{config.epochs} | Train Loss: {loss:.4f}")

    # ── Save Model PKL Checkpoint File ─────────────────────────────────────────
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    pkl_path = Path(f"telugu_2state_{model_name.lower()}.pkl")
    torch.save(
        {
            "model_name": model_name.upper(),
            "scheme": "2STATE",
            "model_state_dict": raw_model.state_dict(),
            "vocab": vocab,
        },
        pkl_path,
    )
    print(f"Saved trained model PKL checkpoint: {pkl_path.resolve()}")

    results = {}
    for split_name in ("train", "val", "test"):
        df_pred = predict_and_generate_rows(model, loaders[split_name], device, model_name, split_name)
        results[split_name] = df_pred

    return results

In [8]:
# Step 6: Dataset Loading, Model Execution & Output Generation
config = Config()

search_paths = [
    config.data_dir / config.excel_filename,
    Path("/kaggle/input/datasets/vallurikeerthiram/testing") / config.excel_filename,
    Path("c:/Users/keert/OneDrive - Amrita vishwa vidyapeetham/3 Phrase Project/phase 2/telugu/training") / config.excel_filename,
    Path(config.excel_filename)
]

target_path = None
for p in search_paths:
    if p.exists():
        target_path = p
        break

if target_path is None:
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".xlsx"):
                target_path = Path(root) / f
                break
        if target_path:
            break

if target_path is None or not target_path.exists():
    raise FileNotFoundError("Dataset Excel file not found. Please verify Kaggle input dataset path.")

print(f"Successfully located dataset Excel file at: {target_path.resolve()}")
wb = pd.ExcelFile(target_path)

sheet_map = {}
for s in wb.sheet_names:
    s_lower = s.lower()
    if "train" in s_lower:
        sheet_map["train"] = s
    elif "val" in s_lower:
        sheet_map["val"] = s
    elif "test" in s_lower:
        sheet_map["test"] = s

splits_df: Dict[str, pd.DataFrame] = {}
for key, sname in sheet_map.items():
    df = wb.parse(sname)
    df["STATE2_LIST"] = df["2state"].map(parse_state2)
    valid_rows = [len(r["STATE2_LIST"]) > 0 for _, r in df.iterrows()]
    splits_df[key] = df.loc[valid_rows].reset_index(drop=True)

print("Building Multi-Vottu 2-State Telugu Abugida vocabularies...")
vocab = TeluguVocab.build(splits_df["train"])

# ── Save Vocabulary PKL File ───────────────────────────────────────────────
vocab_path = Path("telugu_2state_vocab.pkl")
torch.save(vocab, vocab_path)
print(f"Saved vocabulary file: {vocab_path.resolve()}")

loaders: Dict[str, DataLoader] = {}
for split_name, df_split in splits_df.items():
    ds = TeluguAksharaDataset(df_split, vocab)
    loaders[split_name] = DataLoader(
        ds,
        batch_size=config.batch_size,
        shuffle=(split_name == "train"),
        num_workers=config.num_workers,
        collate_fn=collate_telugu_batch,
    )

all_models = ["bilstm", "gru", "rnn", "cnn"]
split_results: Dict[str, List[pd.DataFrame]] = {"train": [], "val": [], "test": []}

for model_name in all_models:
    model_dfs = fit_and_evaluate_model(model_name, vocab, loaders, device, config)
    for sname in ("train", "val", "test"):
        split_results[sname].append(model_dfs[sname])

out_file = Path(config.output_filename)
print(f"\nWriting predictions to Excel file: {out_file.resolve()}")

with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    for sname in ("train", "val", "test"):
        combined_df = pd.concat(split_results[sname], ignore_index=True)
        sheet_title = f"{sname.upper()}_PREDICTIONS"[:31]
        combined_df.to_excel(writer, sheet_name=sheet_title, index=False)

print(f"\nMulti-Vottu 2-State Training & PKL Export completed successfully!")

Successfully located dataset Excel file at: /kaggle/input/datasets/abhaygokavarapu0630/telugu100/telugu_sentence_level_first_100_articles_split.xlsx
Building Multi-Vottu 2-State Telugu Abugida vocabularies...
Saved vocabulary file: /kaggle/working/telugu_2state_vocab.pkl

================ Training BILSTM ================
[Multi-GPU] Wrapping BILSTM with DataParallel across 2 GPUs
[BILSTM] Epoch 1/8 | Train Loss: 0.6349
[BILSTM] Epoch 2/8 | Train Loss: 0.6020
[BILSTM] Epoch 3/8 | Train Loss: 0.5443
[BILSTM] Epoch 4/8 | Train Loss: 0.4673
[BILSTM] Epoch 5/8 | Train Loss: 0.4090
[BILSTM] Epoch 6/8 | Train Loss: 0.3657
[BILSTM] Epoch 7/8 | Train Loss: 0.3388
[BILSTM] Epoch 8/8 | Train Loss: 0.3155
Saved trained model PKL checkpoint: /kaggle/working/telugu_2state_bilstm.pkl

================ Training GRU ================
[Multi-GPU] Wrapping GRU with DataParallel across 2 GPUs
[GRU] Epoch 1/8 | Train Loss: 0.6285
[GRU] Epoch 2/8 | Train Loss: 0.5763
[GRU] Epoch 3/8 | Train Loss: 0.4977
[GRU